# Rotary Positional Embedding (RoPE)

[youtube video](https://www.youtube.com/watch?v=V8r__fXx7tU)

We want a positional encoding that:
* Encodes relative positions naturally
* Preserves dot-product structure in attention
* Can be applied inside attention (Q, K) instead of added to inputs

## Definition: Rotary Positional Embedding (RoPE)


Consider the $2D$ rotational matrix

$$
R(\theta) =
\begin{pmatrix}
\cos(\theta) & -\sin(\theta) \\
\sin(\theta) & \cos(\theta)
\end{pmatrix}
$$

Given angles $\Theta=(\theta_1,\theta_2,\dots,\theta_K)$ whe define the block diagonal matrix
$$
\text{RotD}(\Theta) =
\begin{pmatrix}
R(\theta_1) &  0 & \dots & 0 \\
0 & R(\theta_2) & \dots &0\\
\vdots & \vdots & \ddots& \vdots \\
0 & 0 & \dots &R(\theta_K)\\
\end{pmatrix} \in \mathbb{R}^{2K\times 2K}.
$$

Then the rotationay matrix associated with position $i$ is given by  $\text{RotD}(\Theta_i) \in \mathbb{R}^{d_{\text{emb}}\times d_{\text{emb}}}$ where
 $\Theta_i=i(\omega_1 ,\omega_2 ,\dots,\omega_{d_{\text{emb}/2}})$ and
 $$
\omega_k = \frac{1}{\text{base}^{\frac{2k}{d_{\text{emb}}}}}, \quad k = 1,2,\dots,\frac{d_{\text{emb}}}{2}
$$

 .


### Input

* $X=[x_0|x_1|\dots|x_{\ell-1}]\in \mathbb{R}^{d_{\text{emb}}\times\ell}$

### Parameters

* $ \text{base} \in \mathbb{R} $  (typically $10000$)
* $d_{\text{emb}} \in \mathbb{N} $



### Output

* $$\text{RoPE}(X)=\left[\text{RoPE}(x_0,0)|\text{RoPE}(x_1,1)|\dots|\text{RoPE}(x_{\ell-1},\ell-1)\right] = \left[\text{RotD}(\Theta_0)x_0|\text{RotD}(\Theta_1)x_1|\dots|\text{RotD}(\Theta_{\ell-1})x_{\ell-1}\right]\in \mathbb{R}^{d_{\text{emb}}\times\ell}$$


### Property (relative position)

$$
\langle \text{RoPE}(\mathbf{q}, i), \text{RoPE}(\mathbf{k}, j) \rangle
=
\langle \mathbf{q}, \text{RoPE}(\mathbf{k}, j - i) \rangle
$$

#### Proof:

$$
\begin{align*}
\langle \text{RoPE}(\mathbf{q}, i), \text{RoPE}(\mathbf{k}, j) \rangle &= \text{RoPE}(\mathbf{q}, i)^{T}\cdot \text{RoPE}(\mathbf{k}, j),\\
& = (\text{RotD}(\Theta_i) \mathbf{q})^{T} \cdot \text{RotD}(\Theta_j) \mathbf{k},\\
& =  \mathbf{q}^{T} \text{RotD}(\Theta_i)^{T} \text{RotD}(\Theta_j) \mathbf{k},\\
& =  \mathbf{q}^{T} \text{RotD}(-\Theta_i) \text{RotD}(\Theta_j) \mathbf{k},\\
& =  \mathbf{q}^{T} \text{RotD}(\Theta_j-\Theta_i) \mathbf{k},\\
& =  \mathbf{q}^{T} \text{RotD}(\Theta_{j-i}) \mathbf{k},\\
& =  \mathbf{q}^{T} \text{RoPE}(\mathbf{k}, j-i),\\
& =  \langle \mathbf{q}, \text{RoPE}(\mathbf{k}, j - i) \rangle
\end{align*}
$$

Notice that
$$
\begin{align*}
R(\theta)^{T}&=\begin{pmatrix}
\cos(\theta) & -\sin(\theta) \\
\sin(\theta) & \cos(\theta)
\end{pmatrix}^T=\begin{pmatrix}
\cos(\theta) & \sin(\theta) \\
-\sin(\theta) & \cos(\theta)
\end{pmatrix}
=\begin{pmatrix}
\cos(-\theta) & -\sin(-\theta) \\
\sin(-\theta) & \cos(-\theta)
\end{pmatrix}
= R(-\theta)\\


R(\theta) R(\alpha)&=
\begin{pmatrix}
\cos(\theta) & -\sin(\theta) \\
\sin(\theta) & \cos(\theta)
\end{pmatrix}
\begin{pmatrix}
\cos(\alpha) & -\sin(\alpha) \\
\sin(\alpha) & \cos(\alpha)
\end{pmatrix}
=\begin{pmatrix}
\cos(\theta+\alpha) & -\sin(\theta+\alpha) \\
\sin(\theta+\alpha) & \cos(\theta+\alpha)
\end{pmatrix}
=R(\theta+\alpha)
\end{align*}
$$





### Intuition

* Each pair of features rotates in a 2D plane  
* Position \( p \) controls the rotation angle  
* Different dimensions rotate at different speeds  

👉 Position is encoded as **phase** across multiple frequencies

## Code

In [1]:
import rotary_embedding_torch as rot
import torch

import models.deep_learning.components.embeddings.rope as myrot

In [2]:
rotary = rot.RotaryEmbedding(dim=16, theta=10_000)

dims = {"batch": 1, "heads": 1, "sequence_length": 2, "head_dim": 16}

x = torch.randn(*(dims.values()))
x_rot = rotary.rotate_queries_or_keys(x)

In [3]:
my_rotary = myrot.RotaryEmbedding(dim=16, theta=10_000)
dims = {"batch": 1, "heads": 1, "sequence_length": 2, "head_dim": 16}
myx_rot = my_rotary.rotate_queries_or_keys(x)

In [4]:
torch.norm(myx_rot - x_rot).max()

tensor(2.4074e-07)